In [74]:
import statistics
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Исправления train loop

1. Утечки памяти:
   - не храним в списке тензоры с графом вычислений
   - не вызываем `torch.cuda.empty_cache()` в каждом батче

2. Нарушения асинхронного конвейрера:
   - переносим батчи на GPU через `pin_memory=True` + `non_blocking=True`
   - создаём шум сразу на нужном устройстве
   - используем `optimizer.zero_grad(set_to_none=True)`

3. Нечестные метрики:
   - вместо `time.time()` на GPU используем `torch.cuda.Event`
   - синхронизируемся только для получения корректного времени


In [75]:
def prepare_data() -> TensorDataset:
    x = torch.randn(10000, 128)
    y = torch.randint(0, 2, (10000,))
    return TensorDataset(x, y)


def train():
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = dev.type == "cuda"

    # pinned memory + non_blocking
    dl = DataLoader(
        prepare_data(),
        batch_size=256,
        shuffle=True,
        pin_memory=use_cuda,
    )

    model = nn.Sequential(
        nn.Linear(128, 512), nn.ReLU(),
        nn.Linear(512, 128), nn.ReLU(),
        nn.Linear(128, 2)
    ).to(dev).train()

    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()

    fw_t = []
    bw_t = []
    loss_sum = torch.zeros((), device=dev)

    for x, y in dl:
        x = x.to(dev, non_blocking=use_cuda)
        y = y.to(dev, non_blocking=use_cuda)

        # Шум создаем сразу на нужном устройстве, чтобы не делать лишний CPU -> GPU transfer
        x = x + torch.randn_like(x)
        opt.zero_grad(set_to_none=True)

        s_fw = torch.cuda.Event(enable_timing=True)
        e_fw = torch.cuda.Event(enable_timing=True)
        s_bw = torch.cuda.Event(enable_timing=True)
        e_bw = torch.cuda.Event(enable_timing=True)

        s_fw.record()
        out = model(x)
        loss = crit(out, y)
        e_fw.record()

        s_bw.record()
        loss.backward()
        e_bw.record()

        opt.step()
        torch.cuda.synchronize()

        fw_t.append(s_fw.elapsed_time(e_fw))
        bw_t.append(s_bw.elapsed_time(e_bw))

        # Не сохраняем loss-тензор в список — это держит graph и плодит расход памяти
        # Не используем torch.cuda.empty_cache() в цикле: это только ломает производительность
        loss_sum += loss.detach()

    avg_loss = (loss_sum / len(dl)).item()

    print(
        f"Epoch finished, avg forward time is {statistics.mean(fw_t) / 1000:.13f} ms, "
        f"avg backward time is {statistics.mean(bw_t) / 1000:.13f} ms, "
        f"avg loss is {avg_loss:.4f}"
    )


if __name__ == '__main__':
    train()

Epoch finished, avg forward time is 0.0002924832001 ms, avg backward time is 0.0006440680012 ms, avg loss is 0.6974
